# spaDIVA tutorial: multi-slice P21/P22 spatial ATAC-RNA-seq

This tutorial trains spaDIVA jointly across three postnatal mouse brain sections and visualizes the aligned representation.


## 1. Import packages


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "tutorials" else Path.cwd()
from spaDIVA import cal_spatial, cal_weight, infer_latents, joint_cluster, joint_train_spadiva, lsi

sc.set_figure_params(figsize=(3, 3))
plt.rcParams["figure.dpi"] = 120


## 2. Set random seed


In [ ]:
RANDOM_SEED = 42

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed()
USE_CUDA = False
USE_CUDA


## 3. Load data

Set `DATA_ROOT` to the directory containing the postnatal mouse brain slices.

In [ ]:
DATA_ROOT = Path(os.environ.get("SPADIVA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
DATA_DIR = DATA_ROOT / "datasets" / "P21P22_multi_slice" / "input_data" / "Mouse_postnatal_brain"
path1 = DATA_DIR / "slice1"
path2 = DATA_DIR / "slice2"
path3 = DATA_DIR / "slice3"

adata_omics1_atac = sc.read(path1 / "s1_adata_atac.h5ad")
adata_omics1_rna = sc.read(path1 / "s1_adata_rna.h5ad")
adata_omics2_atac = sc.read(path2 / "s2_adata_atac.h5ad")
adata_omics2_rna = sc.read(path2 / "s2_adata_rna.h5ad")
adata_omics3_atac = sc.read(path3 / "s3_adata_atac.h5ad")
adata_omics3_rna = sc.read(path3 / "s3_adata_rna.h5ad")

adata_omics1_atac.var_names_make_unique()
adata_omics1_rna.var_names_make_unique()
adata_omics2_atac.var_names_make_unique()
adata_omics2_rna.var_names_make_unique()
adata_omics3_atac.var_names_make_unique()
adata_omics3_rna.var_names_make_unique()

adata_omics1_atac.X = adata_omics1_atac.X.astype("float32")
adata_omics1_rna.X = adata_omics1_rna.X.astype("float32")
adata_omics2_atac.X = adata_omics2_atac.X.astype("float32")
adata_omics2_rna.X = adata_omics2_rna.X.astype("float32")
adata_omics3_atac.X = adata_omics3_atac.X.astype("float32")
adata_omics3_rna.X = adata_omics3_rna.X.astype("float32")


## 4. Combine slices


In [ ]:
adata_rna_combined = adata_omics1_rna.concatenate(
    adata_omics2_rna, 
    adata_omics3_rna, 
    batch_key='batch',
)
adata_atac_combined = adata_omics1_atac.concatenate(
    adata_omics2_atac, 
    adata_omics3_atac, 
    batch_key='batch',
)


## 5. Preprocess RNA and ATAC modalities


In [ ]:
def preprocess(adata_omics_rna, adata_omics_atac):
    #rna
    sc.pp.filter_genes(adata_omics_rna, min_cells=1)
    sc.pp.filter_genes(adata_omics_atac, min_cells=1)
    sc.pp.filter_cells(adata_omics_rna, min_genes=1)
    sc.pp.highly_variable_genes(adata_omics_rna, flavor="seurat_v3", n_top_genes=3000)
    adata_omics_rna = adata_omics_rna[:, adata_omics_rna.var.highly_variable]
    sc.pp.normalize_total(adata_omics_rna, target_sum=1e4)
    sc.pp.log1p(adata_omics_rna)
    sc.pp.scale(adata_omics_rna)

    from sklearn.decomposition import  PCA
    pca = PCA(n_components = 64)
    adata_omics_rna.obsm['pca'] = pca.fit_transform(adata_omics_rna.to_df())
    #atac
    adata_omics_atac = adata_omics_atac[adata_omics_rna.obs_names].copy()
    adata_omics_atac.obsm['lsi'] = lsi(adata_omics_atac, use_highly_variable=False, n_components=64 + 1)

    spatial = adata_omics_rna.obsm['spatial']
    return adata_omics_rna, adata_omics_atac


In [ ]:
adata_rna_combined_preprocessed, adata_atac_combined_preprocessed = preprocess(adata_rna_combined, adata_atac_combined)


## 6. Split matrices by slice


In [ ]:
s0 = adata_rna_combined_preprocessed.obs['batch']=='0'
s1 = adata_rna_combined_preprocessed.obs['batch']=='1'
s2 = adata_rna_combined_preprocessed.obs['batch']=='2'
X1_RNA, X1_ATAC, spatial1 = adata_rna_combined_preprocessed[s0].obsm['pca'], adata_atac_combined_preprocessed[s0].obsm['lsi'], adata_atac_combined_preprocessed[s0].obsm['spatial']
X2_RNA, X2_ATAC, spatial2 = adata_rna_combined_preprocessed[s1].obsm['pca'], adata_atac_combined_preprocessed[s1].obsm['lsi'], adata_atac_combined_preprocessed[s1].obsm['spatial']
X3_RNA, X3_ATAC, spatial3 = adata_rna_combined_preprocessed[s2].obsm['pca'], adata_atac_combined_preprocessed[s2].obsm['lsi'], adata_atac_combined_preprocessed[s2].obsm['spatial']


## 7. Build spatial graphs


In [ ]:
edge_index1 = cal_spatial(spatial1, k=4)
edge_index2 = cal_spatial(spatial2, k=4)
edge_index3 = cal_spatial(spatial3, k=4)


## 8. Train multi-slice spaDIVA once


In [ ]:
MAX_EPOCHS = 100
PRE_EPOCH = 0
EPOCHS_PER_UPDATE = 50
LAMBDA_MNN = 0.5
LEARNING_RATE = 1e-3
WEIGHT = 1.0

model, train_loss = joint_train_spadiva(
    X1_ATAC,
    X1_RNA,
    X2_ATAC,
    X2_RNA,
    X3_ATAC,
    X3_RNA,
    edge_index1,
    edge_index2,
    edge_index3,
    learning_rate=LEARNING_RATE,
    weight=WEIGHT,
    max_epochs=MAX_EPOCHS,
    epochs_per_update=EPOCHS_PER_UPDATE,
    pre_epoch=PRE_EPOCH,
    lambda_mnn=LAMBDA_MNN,
    use_cuda=USE_CUDA,
)


## 9. Inspect training loss


In [ ]:
plt.plot(train_loss)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("spaDIVA training loss")
plt.show()


## 10. Infer latent representations


In [ ]:
z1_PoE, z1_atac_loc, z1_rna_loc, w1_atac_loc, w1_rna_loc, x1_atac, x1_rna = infer_latents(model, X1_ATAC, X1_RNA, edge_index=edge_index1, use_cuda=USE_CUDA)
z2_PoE, z2_atac_loc, z2_rna_loc, w2_atac_loc, w2_rna_loc, x2_atac, x2_rna = infer_latents(model, X2_ATAC, X2_RNA, edge_index=edge_index2, use_cuda=USE_CUDA)
z3_PoE, z3_atac_loc, z3_rna_loc, w3_atac_loc, w3_rna_loc, x3_atac, x3_rna = infer_latents(model, X3_ATAC, X3_RNA, edge_index=edge_index3, use_cuda=USE_CUDA)

a1, b1 = cal_weight(z1_atac_loc, z1_rna_loc, k=20)
Z1_WNN = a1.reshape(-1, 1) * z1_atac_loc + b1.reshape(-1, 1) * z1_rna_loc
a2, b2 = cal_weight(z2_atac_loc, z2_rna_loc, k=20)
Z2_WNN = a2.reshape(-1, 1) * z2_atac_loc + b2.reshape(-1, 1) * z2_rna_loc
a3, b3 = cal_weight(z3_atac_loc, z3_rna_loc, k=20)
Z3_WNN = a3.reshape(-1, 1) * z3_atac_loc + b3.reshape(-1, 1) * z3_rna_loc


## 11. Collect aligned outputs


In [ ]:
Z_PoE_all = np.concatenate([z1_PoE, z2_PoE, z3_PoE], axis=0)
Z_ATAC_all = np.concatenate([z1_atac_loc, z2_atac_loc, z3_atac_loc], axis=0)
Z_RNA_all = np.concatenate([z1_rna_loc, z2_rna_loc, z3_rna_loc], axis=0)
W_ATAC_all = np.concatenate([w1_atac_loc, w2_atac_loc, w3_atac_loc], axis=0)
W_RNA_all = np.concatenate([w1_rna_loc, w2_rna_loc, w3_rna_loc], axis=0)
Z_WNN_all = np.concatenate([Z1_WNN, Z2_WNN, Z3_WNN], axis=0)
spatial_all = np.concatenate([spatial1, spatial2, spatial3], axis=0)
slice_labels = np.concatenate([
    np.zeros(z1_PoE.shape[0], dtype=int),
    np.ones(z2_PoE.shape[0], dtype=int),
    np.full(z3_PoE.shape[0], 2, dtype=int),
])
original_indices = np.concatenate([
    np.arange(z1_PoE.shape[0]),
    np.arange(z2_PoE.shape[0]),
    np.arange(z3_PoE.shape[0]),
])

z_adata = sc.AnnData(X=Z_WNN_all)
z_adata.obs["batch"] = slice_labels
z_adata.obs["original_index"] = original_indices
z_adata.obsm["spatial"] = spatial_all
z_adata.obsm["Z"] = Z_WNN_all
z_adata.obsm["Z_poe"] = Z_PoE_all
z_adata.obsm["Z_WNN"] = z_adata.obsm["Z"]
z_adata.obsm["Z_PoE"] = z_adata.obsm["Z_poe"]
z_adata.obsm["Z_ATAC"] = Z_ATAC_all
z_adata.obsm["Z_RNA"] = Z_RNA_all
z_adata.obsm["W_ATAC"] = W_ATAC_all
z_adata.obsm["W_RNA"] = W_RNA_all


## 12. Cluster and visualize aligned domains


In [ ]:
joint_cluster(
    z_adata,
    num_cluster=6,
    title="spaDIVA",
    show=True,
)


## 13. Result object


In [ ]:
z_adata
